In [14]:
import numpy as np
np.random.seed(42)

In [15]:
from indices import *
import tracemalloc
import time

In [16]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

In [17]:
labels = np.concatenate([np.zeros(data[0].shape[1]), np.ones(data[1].shape[1])])
labels.shape

(38978,)

In [18]:
data[1].shape

(12, 19489)

In [19]:
tracemalloc.start()

In [20]:
band_indexes = list(range(1, 12))
encoder = IndicesClassEncoderEq([HueSimp], band_indexes)

feature_id = []
features = []
for i in range(encoder.total_length):
    index = encoder.getIndex(i)
    a = index.args
    if a[1] == a[2]:
        continue

    feature_id.append(i)
    features.append(np.concatenate([index.getValue(data[0]), index.getValue(data[1])]))

features = np.array(features).swapaxes(0, 1)

In [21]:
features.shape

(38978, 1210)

In [22]:
from feature_engine.selection import MRMR

In [23]:
time_start = time.time()

In [24]:
mrmr = MRMR(max_features=3, n_jobs=-1, discrete_features=False, )
mrmr.fit(features, labels)

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

MRMR(discrete_features=False, max_features=3, n_jobs=-1)

In [25]:
time_end = time.time()
print("Time:", time_end - time_start, "sec")
print("MEM usage:", np.array(tracemalloc.get_traced_memory()) / 1024**2, "mb")
tracemalloc.stop()

Time: 276.8953938484192 sec
MEM usage: [ 360.71785069 2159.63970757] mb


In [26]:
mapping = {
    0: "B1",
    1: "B2",
    2: "B3",
    3: "B4",
    4: "B5",
    5: "B6",
    6: "B7",
    7: "B8",
    8: "B8A",
    9: "B9",
    10: "B11",
    11: "B12"
}

selected = np.nonzero(mrmr.get_support())[0]
for id in selected:
    index_id = feature_id[id]
    index = encoder.getIndex(index_id)
    name = getIndexName(index, mapping)
    print("Id:", index_id, "Name:", name)

Id: 144 Name: HueSimp(B3, B4, B3)
Id: 164 Name: HueSimp(B12, B5, B3)
Id: 1274 Name: HueSimp(B11, B7, B12)
